# TrafficGuard AI
## Phase 3: Exploratory Data Analysis (EDA)

### Objectives

- Understand the structure of both datasets.
- Analyze class and dataset split distributions.
- Examine image dimensions and image quality.
- Validate YOLO bounding box annotations.
- Identify missing, corrupted and duplicate images.
- Detect potential data leakage.
- Visualize annotated images.
- Generate statistical reports and visualization graphs.

The findings of this notebook will be used to prepare
high-quality datasets for object detection model training.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import cv2
import yaml
import json
import random
import hashlib
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
from matplotlib.patches import Rectangle

warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(42)
np.random.seed(42)

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 12
})

print("All required libraries imported successfully!")

In [ ]:
%pip install -q pyyaml tqdm seaborn

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

HELMET_SOURCE = (
    PROJECT_ROOT /
    "datasets/raw/helmet/labeled_dataset"
)

SEATBELT_SOURCE = (
    PROJECT_ROOT /
    "datasets/raw/seatbelt"
)

EDA_DIR = PROJECT_ROOT / "results" / "eda"

FIGURES_DIR = EDA_DIR / "figures"
TABLES_DIR = EDA_DIR / "tables"
REPORTS_DIR = EDA_DIR / "reports"

for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

assert HELMET_SOURCE.exists(), "Helmet dataset not found!"
assert SEATBELT_SOURCE.exists(), "Seatbelt dataset not found!"

print("Helmet dataset:", HELMET_SOURCE)
print("Seatbelt dataset:", SEATBELT_SOURCE)

print("\nEDA output directory:")
print(EDA_DIR)

print("\nAll project paths configured successfully!")

In [ ]:
LOCAL_ROOT = Path("/content/trafficguard_eda")

HELMET_DIR = LOCAL_ROOT / "helmet"
SEATBELT_DIR = LOCAL_ROOT / "seatbelt"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def copy_dataset(source, destination, splits):

    destination.mkdir(parents=True, exist_ok=True)

    for split in splits:

        for subfolder in ["images", "labels"]:

            src = source / split / subfolder
            dst = destination / split / subfolder

            if src.exists():

                shutil.copytree(
                    src,
                    dst,
                    dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(
                        ".cache",
                        "*.cache",
                        ".DS_Store"
                    )
                )

                print(f"Copied: {split}/{subfolder}")

            else:
                print(f"Missing: {src}")

    for yaml_file in source.glob("*.yaml"):

        shutil.copy2(
            yaml_file,
            destination / yaml_file.name
        )

print("Copying helmet dataset...")

copy_dataset(
    HELMET_SOURCE,
    HELMET_DIR,
    ["train", "val", "test"]
)

print("\nCopying seatbelt dataset...")

copy_dataset(
    SEATBELT_SOURCE,
    SEATBELT_DIR,
    ["train", "valid", "test"]
)

print("\nLocal dataset preparation completed!")

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".webp", ".bmp"
}

DATASETS = {
    "Helmet": {
        "path": HELMET_DIR,
        "splits": {
            "train": "train",
            "val": "val",
            "test": "test"
        }
    },

    "Seatbelt": {
        "path": SEATBELT_DIR,
        "splits": {
            "train": "train",
            "val": "valid",
            "test": "test"
        }
    }
}

for dataset_name, config in DATASETS.items():

    yaml_path = config["path"] / "data.yaml"

    with open(yaml_path, "r") as file:
        data = yaml.safe_load(file)

    names = data["names"]

    if isinstance(names, list):

        config["classes"] = {
            i: name
            for i, name in enumerate(names)
        }

    else:

        config["classes"] = {
            int(i): name
            for i, name in names.items()
        }

    print("\nDataset:", dataset_name)

    for class_id, class_name in config["classes"].items():

        print(f"Class {class_id}: {class_name}")

In [ ]:
def analyze_file_types(folder):

    file_types = Counter()

    for file in folder.rglob("*"):

        if file.is_file():

            extension = (
                file.suffix.lower()
                or "no_extension"
            )

            file_types[extension] += 1

    return pd.DataFrame(
        file_types.most_common(),
        columns=["File Type", "Count"]
    )

helmet_file_types = analyze_file_types(
    HELMET_SOURCE
)

seatbelt_file_types = analyze_file_types(
    SEATBELT_SOURCE
)

print("HELMET FILE TYPES")
display(helmet_file_types)

print("\nSEATBELT FILE TYPES")
display(seatbelt_file_types)

helmet_file_types.to_csv(
    TABLES_DIR / "helmet_file_types.csv",
    index=False
)

seatbelt_file_types.to_csv(
    TABLES_DIR / "seatbelt_file_types.csv",
    index=False
)

print("\nFile type analysis saved.")

In [ ]:
split_records = []

for dataset_name, config in DATASETS.items():

    for split_name, folder_name in config["splits"].items():

        image_dir = (
            config["path"] /
            folder_name /
            "images"
        )

        label_dir = (
            config["path"] /
            folder_name /
            "labels"
        )

        images = [
            p for p in image_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        labels = list(
            label_dir.rglob("*.txt")
        )

        split_records.append({
            "Dataset": dataset_name,
            "Split": split_name,
            "Images": len(images),
            "Labels": len(labels)
        })

split_df = pd.DataFrame(split_records)

split_df["Percentage"] = (
    split_df["Images"] /
    split_df.groupby("Dataset")["Images"].transform("sum")
    * 100
).round(2)

display(split_df)

split_df.to_csv(
    TABLES_DIR / "dataset_split_distribution.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

colors = [
    "#2563eb",
    "#16a34a",
    "#f59e0b"
]

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    data = split_df[
        split_df["Dataset"] == dataset_name
    ]

    ax.pie(
        data["Images"],
        labels=data["Split"],
        autopct="%1.1f%%",
        startangle=90,
        colors=colors,
        wedgeprops={
            "edgecolor": "white",
            "linewidth": 2
        }
    )

    ax.set_title(
        f"{dataset_name} Dataset\n"
        f"Total Images: {data['Images'].sum():,}"
    )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "01_dataset_split_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Comprehensive Dataset Inspection

Each image and its associated YOLO annotation file
are inspected to identify image dimensions, annotation
quality, missing labels, invalid bounding boxes,
corrupted images and exact duplicate images.

The collected information is stored in structured
dataframes for subsequent statistical analysis.

In [ ]:
image_records = []
object_records = []
issue_records = []

def record_issue(dataset, split, file, issue, detail=""):

    issue_records.append({
        "Dataset": dataset,
        "Split": split,
        "File": str(file),
        "Issue": issue,
        "Detail": str(detail)
    })


def calculate_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        for chunk in iter(
            lambda: file.read(65536),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


def analyze_dataset(dataset_name, config):

    classes = config["classes"]

    for split, folder_name in config["splits"].items():

        image_dir = (
            config["path"] /
            folder_name /
            "images"
        )

        label_dir = (
            config["path"] /
            folder_name /
            "labels"
        )

        images = sorted([
            p for p in image_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ])

        expected_labels = {
            p.relative_to(image_dir).with_suffix(".txt")
            for p in images
        }

        for label in label_dir.rglob("*.txt"):

            if label.relative_to(label_dir) not in expected_labels:

                record_issue(
                    dataset_name,
                    split,
                    label,
                    "Orphan label",
                    "No matching image"
                )

        for image_path in tqdm(
            images,
            desc=f"{dataset_name} - {split}"
        ):

            relative_path = image_path.relative_to(
                image_dir
            )

            label_path = (
                label_dir /
                relative_path.with_suffix(".txt")
            )

            image_id = (
                f"{dataset_name}/"
                f"{split}/"
                f"{relative_path.as_posix()}"
            )

            try:

                with Image.open(image_path) as img:

                    img.verify()

                with Image.open(image_path) as img:

                    width, height = img.size

                    gray = np.array(
                        img.convert("L")
                    )

                brightness = float(gray.mean())

                blur_score = float(
                    cv2.Laplacian(
                        gray,
                        cv2.CV_64F
                    ).var()
                )

                file_hash = calculate_hash(
                    image_path
                )

            except Exception as error:

                record_issue(
                    dataset_name,
                    split,
                    image_path,
                    "Corrupted or unreadable image",
                    error
                )

                continue

            valid_objects = 0
            invalid_objects = 0
            label_status = "valid"

            if not label_path.exists():

                label_status = "missing"

                record_issue(
                    dataset_name,
                    split,
                    image_path,
                    "Missing label"
                )

            else:

                try:

                    lines = label_path.read_text(
                        encoding="utf-8"
                    ).splitlines()

                except Exception as error:

                    lines = []
                    label_status = "unreadable"

                    record_issue(
                        dataset_name,
                        split,
                        label_path,
                        "Unreadable label",
                        error
                    )

                nonempty_lines = [
                    line.strip()
                    for line in lines
                    if line.strip()
                ]

                if not nonempty_lines and label_status == "valid":

                    label_status = "empty"

                    record_issue(
                        dataset_name,
                        split,
                        label_path,
                        "Empty label"
                    )

                seen_boxes = set()

                for line_number, line in enumerate(
                    nonempty_lines,
                    start=1
                ):

                    try:

                        parts = line.split()

                        if len(parts) != 5:
                            raise ValueError(
                                "Expected 5 YOLO values"
                            )

                        class_value = float(parts[0])

                        if not class_value.is_integer():
                            raise ValueError(
                                "Class ID must be an integer"
                            )

                        class_id = int(class_value)

                        x, y, w, h = map(
                            float,
                            parts[1:]
                        )

                        if not np.isfinite(
                            [x, y, w, h]
                        ).all():
                            raise ValueError(
                                "Non-finite coordinates"
                            )

                        if class_id not in classes:
                            raise ValueError(
                                "Unknown class ID"
                            )

                        if not (
                            0 <= x <= 1
                            and 0 <= y <= 1
                            and 0 < w <= 1
                            and 0 < h <= 1
                        ):
                            raise ValueError(
                                "Invalid normalized coordinates"
                            )

                        tolerance = 1e-6

                        if (
                            x - w / 2 < -tolerance
                            or y - h / 2 < -tolerance
                            or x + w / 2 > 1 + tolerance
                            or y + h / 2 > 1 + tolerance
                        ):
                            raise ValueError(
                                "Bounding box outside image"
                            )

                        box_key = (
                            class_id, x, y, w, h
                        )

                        if box_key in seen_boxes:

                            record_issue(
                                dataset_name,
                                split,
                                label_path,
                                "Duplicate annotation",
                                f"Line {line_number}"
                            )

                        seen_boxes.add(box_key)

                        object_records.append({
                            "Dataset": dataset_name,
                            "Split": split,
                            "Image ID": image_id,
                            "Image": image_path.name,
                            "Image Path": str(image_path),
                            "Class ID": class_id,
                            "Class Name": classes[class_id],
                            "X Center": x,
                            "Y Center": y,
                            "Box Width": w,
                            "Box Height": h,
                            "Box Area": w * h,
                            "Box Area Percent": w * h * 100,
                            "Box Width Pixels": w * width,
                            "Box Height Pixels": h * height
                        })

                        valid_objects += 1

                    except Exception as error:

                        invalid_objects += 1

                        record_issue(
                            dataset_name,
                            split,
                            label_path,
                            "Invalid annotation",
                            f"Line {line_number}: {error}"
                        )

            image_records.append({
                "Dataset": dataset_name,
                "Split": split,
                "Image ID": image_id,
                "Image": image_path.name,
                "Image Path": str(image_path),
                "Label Path": str(label_path),
                "Width": width,
                "Height": height,
                "Aspect Ratio": width / height,
                "Megapixels": width * height / 1e6,
                "Brightness": brightness,
                "Blur Score": blur_score,
                "Valid Objects": valid_objects,
                "Invalid Objects": invalid_objects,
                "Label Status": label_status,
                "File Size KB": image_path.stat().st_size / 1024,
                "SHA256": file_hash
            })


print("Dataset analysis engine initialized.")

In [ ]:
# Reset results to avoid duplicate records if rerun.

image_records.clear()
object_records.clear()
issue_records.clear()

for dataset_name, config in DATASETS.items():

    print("\n" + "=" * 55)
    print(f"ANALYZING {dataset_name.upper()} DATASET")
    print("=" * 55)

    analyze_dataset(
        dataset_name,
        config
    )

images_df = pd.DataFrame(image_records)

objects_df = pd.DataFrame(object_records)

issues_df = pd.DataFrame(
    issue_records,
    columns=[
        "Dataset",
        "Split",
        "File",
        "Issue",
        "Detail"
    ]
)

print("\n" + "=" * 55)
print("DATASET ANALYSIS COMPLETED")
print("=" * 55)

print("Readable images:", len(images_df))
print("Valid annotation rows:", len(objects_df))
print("Recorded issues:", len(issues_df))

display(images_df.head())
display(objects_df.head())

In [ ]:
images_df.to_csv(
    TABLES_DIR / "image_inventory.csv",
    index=False
)

objects_df.to_csv(
    TABLES_DIR / "object_annotations.csv",
    index=False
)

issues_df.to_csv(
    TABLES_DIR / "dataset_issues.csv",
    index=False
)

print("Image inventory saved.")
print("Object annotations saved.")
print("Dataset issues saved.")

print("\nSaved to:", TABLES_DIR)

In [ ]:
class_summary = (
    objects_df.groupby(
        ["Dataset", "Class ID", "Class Name"]
    )
    .agg(
        Total_Objects=("Image ID", "size"),
        Images_Containing_Class=("Image ID", "nunique")
    )
    .reset_index()
)

class_summary["Object_Percentage"] = (
    class_summary["Total_Objects"] /
    class_summary.groupby("Dataset")[
        "Total_Objects"
    ].transform("sum")
    * 100
).round(2)

display(class_summary)

class_summary.to_csv(
    TABLES_DIR / "class_distribution.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6)
)

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    subset = class_summary[
        class_summary["Dataset"] == dataset_name
    ]

    bars = ax.bar(
        subset["Class Name"],
        subset["Total_Objects"],
        color=sns.color_palette(
            "Set2",
            len(subset)
        ),
        edgecolor="black",
        linewidth=0.5
    )

    ax.bar_label(
        bars,
        padding=5,
        fmt="%d"
    )

    ax.set_title(
        f"{dataset_name} Class Distribution"
    )

    ax.set_xlabel("Object Class")
    ax.set_ylabel("Annotated Objects")

    ax.tick_params(
        axis="x",
        rotation=20
    )

    ax.margins(y=0.15)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "02_class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
split_class_summary = (
    objects_df.groupby([
        "Dataset",
        "Split",
        "Class Name"
    ])
    .size()
    .reset_index(name="Objects")
)

display(split_class_summary)

split_class_summary.to_csv(
    TABLES_DIR / "class_distribution_by_split.csv",
    index=False
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6)
)

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    subset = split_class_summary[
        split_class_summary["Dataset"] == dataset_name
    ]

    pivot = subset.pivot(
        index="Class Name",
        columns="Split",
        values="Objects"
    ).reindex(
        columns=["train", "val", "test"]
    ).fillna(0)

    pivot.plot(
        kind="bar",
        ax=ax,
        color=[
            "#2563eb",
            "#16a34a",
            "#f59e0b"
        ],
        width=0.75
    )

    ax.set_title(
        f"{dataset_name} Classes by Split"
    )

    ax.set_ylabel("Objects")
    ax.set_xlabel("Class")

    ax.tick_params(
        axis="x",
        rotation=20
    )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "03_classes_by_split.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
imbalance_records = []

TARGET_CLASSES = {
    "Helmet": [0, 1],
    "Seatbelt": [0, 1]
}

for dataset_name, target_ids in TARGET_CLASSES.items():

    subset = objects_df[
        (objects_df["Dataset"] == dataset_name)
        & (objects_df["Split"] == "train")
        & (objects_df["Class ID"].isin(target_ids))
    ]

    counts = (
        subset["Class ID"]
        .value_counts()
        .reindex(target_ids, fill_value=0)
    )

    maximum = counts.max()
    minimum = counts.min()

    ratio = (
        maximum / minimum
        if minimum > 0
        else np.inf
    )

    if minimum == 0:
        status = "Missing target class"

    elif ratio <= 1.5:
        status = "Relatively balanced"

    elif ratio <= 3:
        status = "Moderate imbalance"

    else:
        status = "Significant imbalance"

    imbalance_records.append({
        "Dataset": dataset_name,
        "Majority Class Objects": int(maximum),
        "Minority Class Objects": int(minimum),
        "Imbalance Ratio": round(ratio, 2),
        "Status": status
    })

imbalance_df = pd.DataFrame(
    imbalance_records
)

display(imbalance_df)

imbalance_df.to_csv(
    TABLES_DIR / "class_imbalance_analysis.csv",
    index=False
)

In [ ]:
resolution_summary = (
    images_df.groupby("Dataset")
    .agg(
        Images=("Image", "count"),

        Min_Width=("Width", "min"),
        Max_Width=("Width", "max"),
        Mean_Width=("Width", "mean"),

        Min_Height=("Height", "min"),
        Max_Height=("Height", "max"),
        Mean_Height=("Height", "mean"),

        Mean_Megapixels=("Megapixels", "mean"),
        Mean_Aspect_Ratio=("Aspect Ratio", "mean")
    )
    .round(2)
    .reset_index()
)

display(resolution_summary)

resolution_summary.to_csv(
    TABLES_DIR / "image_resolution_summary.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 10)
)

for row, dataset_name in enumerate(
    DATASETS.keys()
):

    subset = images_df[
        images_df["Dataset"] == dataset_name
    ]

    sns.histplot(
        data=subset,
        x="Width",
        bins=30,
        color="#2563eb",
        ax=axes[row, 0]
    )

    axes[row, 0].set_title(
        f"{dataset_name} Image Width"
    )

    axes[row, 0].set_xlabel("Width (pixels)")

    sns.histplot(
        data=subset,
        x="Height",
        bins=30,
        color="#16a34a",
        ax=axes[row, 1]
    )

    axes[row, 1].set_title(
        f"{dataset_name} Image Height"
    )

    axes[row, 1].set_xlabel("Height (pixels)")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "04_image_resolution_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    subset = images_df[
        images_df["Dataset"] == dataset_name
    ]

    sns.histplot(
        subset["Aspect Ratio"],
        bins=30,
        color="#6366f1",
        ax=ax
    )

    ax.axvline(
        1,
        color="red",
        linestyle="--",
        label="Square Image"
    )

    ax.set_title(
        f"{dataset_name} Aspect Ratios"
    )

    ax.set_xlabel("Width / Height")
    ax.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "05_aspect_ratio_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
bbox_summary = (
    objects_df.groupby([
        "Dataset",
        "Class Name"
    ])
    .agg(
        Objects=("Class ID", "count"),

        Mean_Width_Pixels=(
            "Box Width Pixels", "mean"
        ),

        Mean_Height_Pixels=(
            "Box Height Pixels", "mean"
        ),

        Mean_Area_Percent=(
            "Box Area Percent", "mean"
        ),

        Min_Area_Percent=(
            "Box Area Percent", "min"
        ),

        Max_Area_Percent=(
            "Box Area Percent", "max"
        )
    )
    .round(3)
    .reset_index()
)

display(bbox_summary)

bbox_summary.to_csv(
    TABLES_DIR / "bounding_box_statistics.csv",
    index=False
)

In [ ]:
objects_df["Object Size"] = pd.cut(
    objects_df["Box Area Percent"],
    bins=[0, 1, 5, 100.0001],
    labels=[
        "Small",
        "Medium",
        "Large"
    ],
    include_lowest=True
)

size_summary = (
    objects_df.groupby(
        ["Dataset", "Object Size"],
        observed=False
    )
    .size()
    .reset_index(name="Objects")
)

display(size_summary)

size_summary.to_csv(
    TABLES_DIR / "object_size_distribution.csv",
    index=False
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=size_summary,
    x="Object Size",
    y="Objects",
    hue="Dataset",
    palette=["#2563eb", "#16a34a"]
)

plt.title("Object Size Distribution")
plt.xlabel("Relative Bounding Box Size")
plt.ylabel("Annotated Objects")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "06_object_size_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    subset = objects_df[
        objects_df["Dataset"] == dataset_name
    ]

    sns.scatterplot(
        data=subset,
        x="Box Width",
        y="Box Height",
        hue="Class Name",
        alpha=0.35,
        s=18,
        ax=ax
    )

    ax.set_title(
        f"{dataset_name} Bounding Box Dimensions"
    )

    ax.set_xlabel("Normalized Width")
    ax.set_ylabel("Normalized Height")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "07_bounding_box_dimensions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
quality_summary = (
    images_df.groupby("Dataset")
    .agg(
        Mean_Brightness=("Brightness", "mean"),
        Min_Brightness=("Brightness", "min"),
        Max_Brightness=("Brightness", "max"),
        Mean_Blur_Score=("Blur Score", "mean"),
        Median_Blur_Score=("Blur Score", "median")
    )
    .round(2)
    .reset_index()
)

display(quality_summary)

quality_summary.to_csv(
    TABLES_DIR / "image_quality_summary.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

sns.histplot(
    data=images_df,
    x="Brightness",
    hue="Dataset",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[0]
)

axes[0].set_title(
    "Image Brightness Distribution"
)

axes[0].set_xlabel(
    "Mean Pixel Brightness (0-255)"
)

images_df["Log Blur Score"] = np.log1p(
    images_df["Blur Score"]
)

sns.histplot(
    data=images_df,
    x="Log Blur Score",
    hue="Dataset",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1]
)

axes[1].set_title(
    "Image Sharpness Distribution"
)

axes[1].set_xlabel(
    "Log-transformed Laplacian Variance"
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "09_image_quality_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
quality_flags = []

for dataset_name in DATASETS:

    subset = images_df[
        images_df["Dataset"] == dataset_name
    ].copy()

    blur_threshold = (
        subset["Blur Score"].quantile(0.05)
    )

    subset["Potentially Dark"] = (
        subset["Brightness"] < 40
    )

    subset["Potentially Overexposed"] = (
        subset["Brightness"] > 220
    )

    subset["Low Relative Sharpness"] = (
        subset["Blur Score"] <= blur_threshold
    )

    flagged = subset[
        subset["Potentially Dark"]
        | subset["Potentially Overexposed"]
        | subset["Low Relative Sharpness"]
    ]

    quality_flags.append(flagged)

    print("\nDataset:", dataset_name)

    print(
        "Potentially dark:",
        subset["Potentially Dark"].sum()
    )

    print(
        "Potentially overexposed:",
        subset["Potentially Overexposed"].sum()
    )

    print(
        "Low relative sharpness:",
        subset["Low Relative Sharpness"].sum()
    )

quality_flags_df = pd.concat(
    quality_flags,
    ignore_index=True
)

quality_flags_df.to_csv(
    TABLES_DIR / "image_quality_flags.csv",
    index=False
)

print("\nImage quality flags saved.")

In [ ]:
duplicates_df = images_df[
    images_df.duplicated(
        subset=["Dataset", "SHA256"],
        keep=False
    )
].copy()

duplicates_df = duplicates_df.sort_values(
    ["Dataset", "SHA256", "Split"]
)

duplicate_summary = (
    duplicates_df.groupby("Dataset")
    .agg(
        Images_In_Duplicate_Groups=(
            "Image ID", "count"
        ),

        Duplicate_Groups=(
            "SHA256", "nunique"
        )
    )
    .reset_index()
)

print("EXACT DUPLICATE ANALYSIS")

if duplicates_df.empty:

    print("No exact duplicate files detected.")

else:

    display(duplicate_summary)

    display(
        duplicates_df[
            [
                "Dataset",
                "Split",
                "Image",
                "SHA256"
            ]
        ].head(15)
    )

duplicates_df.to_csv(
    TABLES_DIR / "exact_duplicate_images.csv",
    index=False
)

In [ ]:
split_hash_summary = (
    images_df.groupby(
        ["Dataset", "SHA256"]
    )["Split"]
    .nunique()
    .reset_index(name="Number of Splits")
)

leaking_hashes = split_hash_summary[
    split_hash_summary["Number of Splits"] > 1
]

leakage_df = images_df.merge(
    leaking_hashes,
    on=["Dataset", "SHA256"],
    how="inner"
)

print("=" * 55)
print("EXACT CROSS-SPLIT DATA LEAKAGE")
print("=" * 55)

print(
    "Duplicate groups across splits:",
    len(leaking_hashes)
)

print(
    "Affected images:",
    len(leakage_df)
)

if leakage_df.empty:

    print(
        "\nNo exact duplicate files "
        "found across dataset splits."
    )

else:

    display(
        leakage_df[
            [
                "Dataset",
                "Split",
                "Image",
                "Number of Splits"
            ]
        ].head(20)
    )

leakage_df.to_csv(
    TABLES_DIR / "cross_split_leakage.csv",
    index=False
)

In [ ]:
%pip install -q ImageHash

In [ ]:
import imagehash
from itertools import combinations

perceptual_records = []

for record in tqdm(
    images_df.to_dict("records"),
    desc="Calculating perceptual hashes"
):

    try:

        with Image.open(
            record["Image Path"]
        ) as img:

            phash = imagehash.phash(
                img,
                hash_size=8
            )

        perceptual_records.append({
            "Dataset": record["Dataset"],
            "Split": record["Split"],
            "Image ID": record["Image ID"],
            "Image": record["Image"],
            "Image Path": record["Image Path"],
            "SHA256": record["SHA256"],
            "Perceptual Hash": str(phash)
        })

    except Exception as error:

        print(
            "Hash error:",
            record["Image"],
            error
        )

phash_df = pd.DataFrame(
    perceptual_records
)

phash_df.to_csv(
    TABLES_DIR / "perceptual_hashes.csv",
    index=False
)

print(
    "Perceptual hashes generated:",
    len(phash_df)
)

In [ ]:
# A 64-bit hash is represented by a hexadecimal string.
# Hamming distance measures the number of differing bits.

near_duplicate_records = []

HASH_THRESHOLD = 5

for dataset_name in DATASETS:

    subset = phash_df[
        phash_df["Dataset"] == dataset_name
    ]

    groups = {
        split: group.to_dict("records")
        for split, group in subset.groupby("Split")
    }

    for split_a, split_b in combinations(
        groups.keys(),
        2
    ):

        records_a = groups[split_a]
        records_b = groups[split_b]

        hashed_b = [
            (
                row,
                int(row["Perceptual Hash"], 16)
            )
            for row in records_b
        ]

        for row_a in tqdm(
            records_a,
            desc=f"{dataset_name}: {split_a} vs {split_b}"
        ):

            hash_a = int(
                row_a["Perceptual Hash"],
                16
            )

            for row_b, hash_b in hashed_b:

                if row_a["SHA256"] == row_b["SHA256"]:
                    continue

                distance = (
                    hash_a ^ hash_b
                ).bit_count()

                if distance <= HASH_THRESHOLD:

                    near_duplicate_records.append({
                        "Dataset": dataset_name,
                        "Split A": split_a,
                        "Split B": split_b,
                        "Image A": row_a["Image"],
                        "Image B": row_b["Image"],
                        "Path A": row_a["Image Path"],
                        "Path B": row_b["Image Path"],
                        "Hamming Distance": distance
                    })

near_duplicates_df = pd.DataFrame(
    near_duplicate_records,
    columns=[
        "Dataset",
        "Split A",
        "Split B",
        "Image A",
        "Image B",
        "Path A",
        "Path B",
        "Hamming Distance"
    ]
)

near_duplicates_df.to_csv(
    TABLES_DIR / "potential_near_duplicates.csv",
    index=False
)

print(
    "\nPotential cross-split near-duplicate pairs:",
    len(near_duplicates_df)
)

display(near_duplicates_df.head(15))

In [ ]:
if near_duplicates_df.empty:

    print("No potential near-duplicate pairs found.")

else:

    samples = near_duplicates_df.sort_values(
        "Hamming Distance"
    ).head(5)

    fig, axes = plt.subplots(
        len(samples),
        2,
        figsize=(12, 4 * len(samples)),
        squeeze=False
    )

    for i, (_, row) in enumerate(
        samples.iterrows()
    ):

        with Image.open(row["Path A"]) as img:
            axes[i, 0].imshow(img.convert("RGB"))

        with Image.open(row["Path B"]) as img:
            axes[i, 1].imshow(img.convert("RGB"))

        axes[i, 0].set_title(
            f"{row['Dataset']} - {row['Split A']}"
        )

        axes[i, 1].set_title(
            f"{row['Dataset']} - {row['Split B']}\n"
            f"Distance: {row['Hamming Distance']}"
        )

        axes[i, 0].axis("off")
        axes[i, 1].axis("off")

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / "10_near_duplicate_examples.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
if issues_df.empty:

    print("No structural dataset issues detected.")

    issue_summary = pd.DataFrame(
        columns=["Dataset", "Issue", "Count"]
    )

else:

    issue_summary = (
        issues_df.groupby([
            "Dataset",
            "Issue"
        ])
        .size()
        .reset_index(name="Count")
    )

    display(issue_summary)

    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=issue_summary,
        y="Issue",
        x="Count",
        hue="Dataset",
        palette=["#2563eb", "#16a34a"]
    )

    plt.title("Dataset Quality Issues")
    plt.xlabel("Number of Recorded Issues")
    plt.ylabel("Issue Type")

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / "11_dataset_quality_issues.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

issue_summary.to_csv(
    TABLES_DIR / "dataset_issue_summary.csv",
    index=False
)

In [ ]:
object_count_summary = (
    images_df.groupby("Dataset")
    .agg(
        Min_Objects=("Valid Objects", "min"),
        Max_Objects=("Valid Objects", "max"),
        Mean_Objects=("Valid Objects", "mean"),
        Median_Objects=("Valid Objects", "median")
    )
    .round(2)
    .reset_index()
)

display(object_count_summary)

object_count_summary.to_csv(
    TABLES_DIR / "objects_per_image_summary.csv",
    index=False
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

for ax, dataset_name in zip(
    axes,
    DATASETS.keys()
):

    subset = images_df[
        images_df["Dataset"] == dataset_name
    ]

    sns.histplot(
        subset["Valid Objects"],
        discrete=True,
        ax=ax,
        color="#2563eb"
    )

    ax.set_title(
        f"{dataset_name}: Objects per Image"
    )

    ax.set_xlabel("Annotated Objects")
    ax.set_ylabel("Number of Images")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "12_objects_per_image.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
CLASS_COLORS = {
    "with-helmet": "#16a34a",
    "without-helmet": "#dc2626",
    "person-seatbelt": "#16a34a",
    "person-noseatbelt": "#dc2626",
    "seatbelt": "#2563eb",
    "windshield": "#f59e0b"
}

def draw_annotated_image(ax, image_record):

    image_path = image_record["Image Path"]

    with Image.open(image_path) as file:

        image = file.convert("RGB")

    width, height = image.size

    ax.imshow(image)

    annotations = objects_df[
        objects_df["Image ID"]
        == image_record["Image ID"]
    ]

    for _, row in annotations.iterrows():

        x = row["X Center"]
        y = row["Y Center"]
        w = row["Box Width"]
        h = row["Box Height"]

        xmin = (x - w / 2) * width
        ymin = (y - h / 2) * height

        box_width = w * width
        box_height = h * height

        class_name = row["Class Name"]

        color = CLASS_COLORS.get(
            class_name,
            "#2563eb"
        )

        rect = Rectangle(
            (xmin, ymin),
            box_width,
            box_height,
            linewidth=2,
            edgecolor=color,
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            xmin,
            max(0, ymin - 3),
            class_name,
            color="white",
            fontsize=8,
            bbox={
                "facecolor": color,
                "alpha": 0.9,
                "pad": 2,
                "edgecolor": "none"
            }
        )

    ax.set_title(
        f"{image_record['Split']} | "
        f"{len(annotations)} objects",
        fontsize=10
    )

    ax.axis("off")


def visualize_dataset(dataset_name, count=9):

    subset = images_df[
        (images_df["Dataset"] == dataset_name)
        & (images_df["Valid Objects"] > 0)
    ]

    if subset.empty:

        print("No annotated images available.")
        return

    samples = subset.sample(
        n=min(count, len(subset)),
        random_state=42
    )

    fig, axes = plt.subplots(
        3, 3,
        figsize=(16, 14)
    )

    fig.suptitle(
        f"{dataset_name} Dataset - YOLO Annotations",
        fontsize=18,
        fontweight="bold"
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        samples.iterrows()
    ):

        draw_annotated_image(
            ax,
            row
        )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"13_{dataset_name.lower()}_annotated_samples.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


visualize_dataset("Helmet")
visualize_dataset("Seatbelt")

In [ ]:
final_records = []

for dataset_name in DATASETS:

    dataset_images = images_df[
        images_df["Dataset"] == dataset_name
    ]

    dataset_objects = objects_df[
        objects_df["Dataset"] == dataset_name
    ]

    dataset_issues = issues_df[
        issues_df["Dataset"] == dataset_name
    ]

    dataset_duplicates = duplicates_df[
        duplicates_df["Dataset"] == dataset_name
    ]

    dataset_leakage = leakage_df[
        leakage_df["Dataset"] == dataset_name
    ]

    dataset_near_duplicates = near_duplicates_df[
        near_duplicates_df["Dataset"] == dataset_name
    ]

    final_records.append({

        "Dataset": dataset_name,

        "Total Images Discovered": int(
            split_df.loc[
                split_df["Dataset"] == dataset_name,
                "Images"
            ].sum()
        ),

        "Readable Images": len(dataset_images),

        "Valid Annotation Rows": len(dataset_objects),

        "Declared Classes": len(
            DATASETS[dataset_name]["classes"]
        ),

        "Observed Classes": (
            dataset_objects["Class ID"].nunique()
        ),

        "Corrupted Images": int(
            (
                dataset_issues["Issue"]
                == "Corrupted or unreadable image"
            ).sum()
        ),

        "Missing Labels": int(
            (
                dataset_issues["Issue"]
                == "Missing label"
            ).sum()
        ),

        "Empty Labels": int(
            (
                dataset_issues["Issue"]
                == "Empty label"
            ).sum()
        ),

        "Unreadable Labels": int(
            (
                dataset_issues["Issue"]
                == "Unreadable label"
            ).sum()
        ),

        "Orphan Labels": int(
            (
                dataset_issues["Issue"]
                == "Orphan label"
            ).sum()
        ),

        "Invalid Annotations": int(
            (
                dataset_issues["Issue"]
                == "Invalid annotation"
            ).sum()
        ),

        "Images In Exact Duplicate Groups": len(
            dataset_duplicates
        ),

        "Exact Cross-Split Leakage Images": len(
            dataset_leakage
        ),

        "Potential Near-Duplicate Pairs": len(
            dataset_near_duplicates
        ),

        "Mean Image Width": round(
            dataset_images["Width"].mean(), 2
        ),

        "Mean Image Height": round(
            dataset_images["Height"].mean(), 2
        ),

        "Mean Objects per Image": round(
            dataset_images["Valid Objects"].mean(), 2
        )
    })

final_eda_df = pd.DataFrame(
    final_records
)

display(final_eda_df.T)

final_eda_df.to_csv(
    TABLES_DIR / "final_eda_summary.csv",
    index=False
)

In [ ]:
report_lines = [
    "TRAFFICGUARD AI",
    "PHASE 3 - EXPLORATORY DATA ANALYSIS",
    "=" * 65,
    ""
]

for record in final_records:

    report_lines.append(
        record["Dataset"].upper() + " DATASET"
    )

    report_lines.append("-" * 65)

    for key, value in record.items():

        if key != "Dataset":

            report_lines.append(
                f"{key}: {value}"
            )

    report_lines.append("")

report_lines.append("CLASS IMBALANCE")
report_lines.append("-" * 65)

report_lines.append(
    imbalance_df.to_string(index=False)
)

report_lines.append("")

report_lines.append("DEVELOPMENT NOTES")
report_lines.append("-" * 65)

report_lines.extend([
    "Raw datasets remain unchanged.",
    "Original class mappings are preserved.",
    "Potential quality issues require review.",
    "Potential near-duplicates require confirmation.",
    "Dataset cleaning will be performed in Phase 4.",
    "Final YAML configurations will be generated",
    "after preprocessing and class remapping."
])

report_text = "\n".join(report_lines)

report_path = (
    REPORTS_DIR / "EDA_analysis_report.txt"
)

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print(report_text)

print("\nReport saved at:")
print(report_path)

In [ ]:
print("=" * 65)
print("TRAFFICGUARD AI - PHASE 3 OUTPUTS")
print("=" * 65)

for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR
]:

    print(f"\n{folder.name.upper()}")

    files = sorted([
        p for p in folder.iterdir()
        if p.is_file()
    ])

    for file in files:

        print(
            f" - {file.name} "
            f"({file.stat().st_size / 1024:.1f} KB)"
        )

    print("Total files:", len(files))

print("\n" + "=" * 65)

print("PHASE 3 ANALYSIS GENERATED")
print("Review required before preprocessing.")

print("=" * 65)